# SEACells baselines: `arbf` (original) + `bk08` affinity, N=200 (s28nsc)

Standalone notebook, separated out from `ctx_pca_product_experiment.ipynb`
because SEACells archetypal analysis on this dataset (58423 cells) is slow and
crash-prone -- keeping it isolated means a crash here doesn't force re-running
the scProto training/eval cells too.

Builds two graphs (`arbf` -- standard, unmodified SEACells recipe on plain PCA;
`bk08` -- real pybanksy lambda=0.8 embedding + real SEACells kernel, the SAME
graph scProto trains on in the other notebook) and runs SEACells archetypal
analysis on each, `N_SEACELLS=200`.

Everything saves to disk keyed by `(ds_id, tag)` regardless of which notebook
calls it -- the comparison/eval cell in `ctx_pca_product_experiment.ipynb`
(`full-eval-code`) will pick up these results the moment they exist on disk, no
need to duplicate the comparison here.

**Root cause of the earlier crash**: SEACells only supports `use_sparse` on
CPU. The old backend-selection logic defaulted to GPU+dense whenever a GPU was
available, which for 58423 cells means materializing a ~58423x58423 dense
matrix (tens of GB) -- guaranteed OOM regardless of other settings. Fixed in
`metric_helpers/metacell_metrics.py`'s `_seacells_backend()`: now forces
CPU+sparse once `n_cells > 30000`, even with a GPU present.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Same numpy-safe install sequence as plan1_niche_recovery_eval.ipynb /
# fibroblast_scproto_vs_seacell.ipynb -- proven working in this repo. Do NOT add
# --upgrade/--force-reinstall to the SEACells line -- tried that, it breaks
# anndata/zarr (AttributeError: module numpy.dtypes has no attribute
# StringDType). Plain install + this exact numpy/scipy pin sequence is what
# actually works.
!pip install -q scarches faiss-cpu scib-metrics
!pip install git+https://github.com/dpeerlab/SEACells.git --quiet --no-deps
!pip install -q pybanksy
!pip install numpy scipy --upgrade -q
!pip install -q palantir harmonypy statsmodels leidenalg python-igraph
!pip install -q "numpy==1.26.4" "scipy==1.13.1"
!pip install --upgrade --force-reinstall numpy cupy-cuda12x
!pip install "numpy<2.3"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
seacells 0.3.3 requires pyranges, which is not installed.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.5 which is incompatible.
pytensor 2.38.3 requires numba<=0.65.1,>=0.58, but you have numba 0.66.0 which is incompatible.
  Preparing metadata (setup.py) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
seacells 0.3.3 requires pyranges, which is not installed.
numba 0.66.0 requires numpy<2.5,>=1.22, but you have numpy 2.5.1 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.5 which is incompatible.
pytensor 2.38.3 requires numba<=0.65.1,>=0.58, but you have numba 0.66.0 which is incompatible.
ERROR: pip's dependency resol

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 103.2 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.5.1
    Uninstalling numpy-2.5.1:
      Successfully uninstalled numpy-2.5.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
seacells 0.3.3 requires pyranges, which is not installed.
anndata 0.13.2 requires scipy!=1.17.0,>=1.14, but you have scipy 1.13.1 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.5 which is incompatible.
access 1.1.10.post3 requires scipy>=1.14.1, but you have scipy 1.13.1 which is incompatible.
pytensor 2.38.3 requires numba<=0.65.1,>=0.58, but you have numba 0.66.0 which is incompatible.
tsfresh 0.21.2 requires scipy>=1.14.0; python_version >= "3.10", but you have scipy

**IMPORTANT: restart the runtime now** (Runtime -> Restart session) before running the cells below.

In [1]:
%run /content/drive/MyDrive/codes/interpretable-prototype/notebooks/nb_setup.py

nb_setup done. Available: get_trainer, run_mc_task, fig_*, LAMBDA_PROTO_UMAP, LAMBDA_PROTO_UMAP_PRECON, LAMBDA_PARAM_UMAP, LAMBDA_RECON_ONLY, train_sure, eval_sure_task1/2/3
Configs: {'LAMBDA_PROTO_UMAP': {'lambda_umap': 1, 'lambda_swav': 0, 'lambda_kl': 0, 'lambda_recon': 0, 'lambda_proto_recon': 0.0, 'umap_similarity': 'proto'}, 'LAMBDA_PARAM_UMAP': {'lambda_umap': 1, 'lambda_swav': 0, 'lambda_kl': 0, 'lambda_recon': 0, 'lambda_proto_recon': 0.0, 'umap_similarity': 'embedding'}, 'LAMBDA_RECON_ONLY': {'lambda_umap': 0, 'lambda_swav': 0, 'lambda_kl': 0, 'lambda_recon': 1, 'lambda_proto_recon': 0.0}}


In [2]:
import os
import pickle

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import scipy.sparse as sp

from interpretable_ssl.datasets.dataset_configs import DATASETS
from interpretable_ssl.configs.paths import CODE_DIR, get_affinity_path
from interpretable_ssl.augmenters.graph_generator import generate_affinity
from interpretable_ssl.evaluation.batch_correct_baselines import run_seacells_on_latent

DS_ID = 's28nsc'
CT_KEY = 'celltypes'
NICHE_KEY = 'niches_2D'
BATCH_KEY = DATASETS[DS_ID].get('batch_key')
K_GRAPH = 50
N_SEACELLS = 200

pd.set_option('display.max_columns', 50)
plt.rcParams['figure.dpi'] = 150

## Load dataset (no ground truth needed here -- that lives in the scProto notebook, only used for eval, not for building graphs/running SEACells)

In [3]:
ds_conf = DATASETS[DS_ID]
adata = sc.read_h5ad(ds_conf['path'])
print(adata)

AnnData object with n_obs × n_vars = 58423 × 960
    obs: 'section', 'celltypes', 'niches_3D', 'niches_2D', 'fibroblast_subclusters', 'tumor_pseudotime_rank', 'EMT_niche'
    uns: 'EMT_niche_colors', 'celltypes_colors', 'fibroblast_subclusters_colors', 'log1p', 'niches_2D_colors', 'niches_3D_colors', 'pca'
    obsm: 'X_covet', 'X_ctx', 'X_pca', 'X_umap_2D_neighbourhoods', 'X_umap_3D_neighbourhoods', 'X_umap_SCT', 'spatial'
    varm: 'PCs'
    layers: 'SCT', 'counts', None (.X)


## Build `arbf` and `bk08` graphs (cached -- instant if already built by the other notebook)

In [4]:
graphs = {}

for gname in ['arbf', 'bk08']:
    cache_path = get_affinity_path(DS_ID, adata.n_obs, k_neighbors=K_GRAPH, affinity_type=gname)
    if os.path.exists(cache_path):
        print(f'=== {gname}: loading cached graph from {cache_path} ===')
        with open(cache_path, 'rb') as f:
            graphs[gname] = pickle.load(f)
    else:
        print(f'=== {gname}: no cache at {cache_path}, building ===')
        graphs[gname] = generate_affinity(adata, k=K_GRAPH, bk=BATCH_KEY, affinity_type=gname)
        os.makedirs(os.path.dirname(cache_path), exist_ok=True)
        with open(cache_path, 'wb') as f:
            pickle.dump(graphs[gname], f)
        print(f'Saved to {cache_path} for future runs.')


=== arbf: loading cached graph from ./graphs/affinity_s28nsc58423_ncomp50_kneighbors50_arbf.pkl ===
=== bk08: loading cached graph from ./graphs/affinity_s28nsc58423_ncomp50_kneighbors50_bk08.pkl ===


## Run SEACells archetypal analysis on each graph

`run_seacells_on_latent` has its own on-disk cache (`skip_if_exists=True` by
default -- checks `metrics.json` + matching `N_SEACELLS`), so if a previous
attempt actually completed and saved before crashing, this will skip straight
to loading it instead of recomputing. Run one graph at a time (separate cells)
so a crash on one doesn't lose progress on the other.

In [5]:
print('=== SEACells archetypal on arbf ===')
seacell_arbf = run_seacells_on_latent(DS_ID, adata, n_seacells=N_SEACELLS, aff=graphs['arbf'], tag='arbf')
print('save_path:', seacell_arbf['save_path'])

=== SEACells archetypal on arbf ===
[waypoint init] N=58423  k=200  n_eigs=10  nnz=4352008  nnz/row=74.5
[waypoint init] computing diffusion map ...
[waypoint init] diffusion map done — 10 eigenvectors


waypoint MaxMin: 100%|██████████| 199/199 [00:00<00:00, 707.32archetype/s]


[waypoint init] selected 200 archetype seed cells
[SEACells backend] No GPU → use_sparse=True (sparse CPU, avoids dense K)
[SEACells backend] using our own optimized sparse-CPU SEACells (never materializes kernel_matrix @ kernel_matrix.T)
Welcome to SEACells!
Using provided list of initial archetypes
Randomly initialized A matrix.
Setting convergence threshold at 541.53698
Starting iteration 1.
Completed iteration 1.
Starting iteration 10.
Completed iteration 10.
Converged after 16 iterations.


100%|██████████| 200/200 [00:06<00:00, 29.40it/s]


saving to:  /content/drive/MyDrive/models/s28nsc/seacell_arbf
  delta kept: X=no (deduped), 1 layer(s), 0 varm, 0 obsm, 0 obsp, obs cols ['SEACell']
  saved soft_assignments.npz (58423, 200) to /content/drive/MyDrive/models/s28nsc/seacell_arbf
Loading SEACell from /content/drive/MyDrive/models/s28nsc/seacell_arbf ...
[seacell] unused protos: 0/200 (0.00%)
[seacell] mean cell-type purity: 0.5396  (size-weighted: 0.4968 ± 0.1948)
[seacell] mean niche purity: 0.7396  (size-weighted: 0.6473 ± 0.1884)
[seacell] mean batch entropy: -0.0000  (size-weighted: -0.0000 ± 0.0000)
[seacell] coverage: 0.7222
[seacell] modularity: 0.4062
[seacell] per-batch modularity: mean=0.4062, std=nan
[aff_dc_compactness] looking for graph at: ./graphs/affinity_s28nsc58423_ncomp50_kneighbors50_arbf.pkl
[aff_dc_compactness] mean=0.1136 | saved to /content/drive/MyDrive/models/s28nsc/seacell_arbf/aff_dc_compactness.csv
[seacell] saved metrics to /content/drive/MyDrive/models/s28nsc/seacell_arbf
SEACell UMAP data s

  0%|          | 0/1 [00:00<?, ?it/s]

Deleted: tmp_7c7b220a.h5ad
[seacell task2] coverage: 0.7222
[seacell task2] scgraph_corr_avg: 0.8398
[seacell task2] scgraph_corr_std: 0.0902
save_path: /content/drive/MyDrive/models/s28nsc/seacell_arbf


In [6]:
print('=== SEACells archetypal on bk08 ===')
seacell_bk08 = run_seacells_on_latent(DS_ID, adata, n_seacells=N_SEACELLS, aff=graphs['bk08'], tag='bk08')
print('save_path:', seacell_bk08['save_path'])

=== SEACells archetypal on bk08 ===
[waypoint init] N=58423  k=200  n_eigs=10  nnz=4058285  nnz/row=69.5
[waypoint init] computing diffusion map ...
[waypoint init] diffusion map done — 10 eigenvectors


waypoint MaxMin: 100%|██████████| 199/199 [00:00<00:00, 638.25archetype/s]


[waypoint init] selected 200 archetype seed cells
[SEACells backend] No GPU → use_sparse=True (sparse CPU, avoids dense K)
[SEACells backend] using our own optimized sparse-CPU SEACells (never materializes kernel_matrix @ kernel_matrix.T)
Welcome to SEACells!
Using provided list of initial archetypes
Randomly initialized A matrix.
Setting convergence threshold at 632.29227
Starting iteration 1.
Completed iteration 1.
Starting iteration 10.
Completed iteration 10.
Converged after 14 iterations.


100%|██████████| 200/200 [00:06<00:00, 30.64it/s]


saving to:  /content/drive/MyDrive/models/s28nsc/seacell_bk08
  delta kept: X=no (deduped), 1 layer(s), 0 varm, 0 obsm, 1 obsp, obs cols ['SEACell']
  saved soft_assignments.npz (58423, 200) to /content/drive/MyDrive/models/s28nsc/seacell_bk08
Loading SEACell from /content/drive/MyDrive/models/s28nsc/seacell_bk08 ...
[seacell] unused protos: 0/200 (0.00%)
[seacell] mean cell-type purity: 0.5307  (size-weighted: 0.5127 ± 0.2185)
[seacell] mean niche purity: 0.7454  (size-weighted: 0.7033 ± 0.1852)
[seacell] mean batch entropy: -0.0000  (size-weighted: -0.0000 ± 0.0000)
[seacell] coverage: 0.6667
[seacell] modularity: 0.3518
[seacell] per-batch modularity: mean=0.3518, std=nan
[aff_dc_compactness] looking for graph at: ./graphs/affinity_s28nsc58423_ncomp50_kneighbors50_arbf.pkl
[aff_dc_compactness] mean=0.1591 | saved to /content/drive/MyDrive/models/s28nsc/seacell_bk08/aff_dc_compactness.csv
[seacell] saved metrics to /content/drive/MyDrive/models/s28nsc/seacell_bk08
SEACell UMAP data s

  0%|          | 0/1 [00:00<?, ?it/s]

Deleted: tmp_4e82f0f3.h5ad
[seacell task2] coverage: 0.6667
[seacell task2] scgraph_corr_avg: 0.8553
[seacell task2] scgraph_corr_std: 0.1126
save_path: /content/drive/MyDrive/models/s28nsc/seacell_bk08


## Done

Both `save_path`s above are what `ctx_pca_product_experiment.ipynb`'s
`full-eval-code` cell needs (`seacell_results['arbf']['save_path']` /
`seacell_results['bk08']['save_path']`) -- go back to that notebook and run the
full-comparison cell; it'll find these on disk.